In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers pypdf accelerate --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 82.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.9/328.9 kB 24.2 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pypdf import PdfReader
import faiss
import numpy as np
import os

In [ ]:
import os
from pypdf import PdfReader

def load_documents(folder):
    docs = []

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        if file.endswith(".pdf"):
            reader = PdfReader(path)
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    docs.append(text)

        elif file.endswith(".txt"):
            with open(path, "r", encoding="utf-8") as f:
                docs.append(f.read())

    return docs

documents = load_documents("source")
print("Documents loaded:", len(documents))
print(documents[0][:400])

Documents loaded: 4
RAG
We use Rag because your llm hallucinate it can give wrong information so in order to combat that we usually do if I want my llm to be trained on some particular domain specific data I fine tune it. Fine tuning means I update weights and parameters and we studied that by FT it is computationally expansive we  update particular weights and layers no everyone.
Now we say we connnect external db a


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
def chunk_text(text, chunk_size=350, overlap=60):
    tokens = tokenizer.encode(text)
    chunks = []

    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]

        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)

        start += chunk_size - overlap # overlapping 10 chunks

    return chunks

chunks = [chunk for doc in documents for chunk in chunk_text(doc)]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (583 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
print("Length of chunks:",len(chunks))
for chunk in chunks:
  print("{",chunk,"}")


Length of chunks: 35
{ RAG We use Rag because your llm hallucinate it can give wrong information so in order to combat that we usually do if I want my llm to be trained on some particular domain specific data I fine tune it. Fine tuning means I update weights and parameters and we studied that by FT it is computationally expansive we update particular weights and layers no everyone. Now we say we connnect external db and external knowledge. Now if I }
{ db and external knowledge. Now if I want any updated or domain specific information so my Llm takes external information and generate and give to me, this is RAG So RAG is one criteria to solve hallucination So if we are asked why hallucination occurs. If our data is outdated, factually not correct, can be web search but not this, lack of grounding means generating based on probability of what it been trained not buying gradient }
{ on probability of what it been trained not buying gradient, no fact checker, training data is old and now

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

chunk_embeddings = embedder.encode(
    chunks, convert_to_numpy=True, show_progress_bar=True
).astype("float32")

faiss.normalize_L2(chunk_embeddings)
chunk_embeddings.shape

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(35, 384)

In [ ]:
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print("FAISS index size:", index.ntotal)

FAISS index size: 35


In [ ]:
def retrieve(query, top_k=5):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(distances[0], indices[0]):
        results.append({"text":chunks[idx], "score": float(score)})
    return results

In [ ]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [ ]:
generator = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(device)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def gtAnswer(query, topK=5):
    retrieved = retrieve(query, topK)

    context = "\n\n".join([r["text"] for r in retrieved])


    prompt = f"""
Answer the question based in the context only

Question: {query}

Context:
{context}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = generator.generate(**inputs, max_new_tokens=150)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, retrieved

In [ ]:
query = "What does it say about context window?"
answer, retrieved_chunks = gtAnswer(query)

print("Answer:\n", answer)
print("\nRertrieved Chunks:")
for r in retrieved_chunks:
    print("-", r["text"][:120].replace("\n"," "), "...")

Answer:
 limited

Rertrieved Chunks:
- a certain constraint they take data of certain context at one time. Like in mistral 4096 was it's context windows. If we ...
- are not sure about it don't give me a comprehensive answer, so it gives story by its own. So what ever you added in prom ...
- , when a user prompts an llm that question turn to embidding and it's similarity is searched which has been stored in th ...
- whenever user query comes embedding comes in its place and second one is context that is top k. Then we feed this in llm ...
- s sementic it is just experiment level, it is not very authentic as it is still on experiment level. Chunk overlap: numb ...


In [ ]:
import re

def normalize(text):
    if text is None: return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    return " ".join(text.split())

In [ ]:
def evaluate_recall_precision(retrieved_chunks, ground_truth, all_chunks):
    gt = normalize(ground_truth)

    all_relevant_chunks = [
        normalize(c) for c in all_chunks if gt in normalize(c)
    ]
    total_relevant = len(all_relevant_chunks)

    retrieved_texts = [normalize(r["text"]) for r in retrieved_chunks]

    relevant_found = 0
    for rel in all_relevant_chunks:
        for ret in retrieved_texts:
            if rel in ret:
                relevant_found += 1
                break

    total_retrieved = len(retrieved_chunks)

    recall = relevant_found / total_relevant if total_relevant > 0 else 0
    precision = relevant_found / total_retrieved

    return recall, precision, total_relevant

In [ ]:
def hallucination_check(answer, retrieved_chunks):
    ans = normalize(answer)
    ctx = " ".join([normalize(r["text"]) for r in retrieved_chunks])

    missing_words = [w for w in ans.split() if w not in ctx]
    hallucinated = len(missing_words) > 3

    return hallucinated, missing_words[:20]

In [ ]:
def evaluate_rag(query, ground_truth, topk=5):
    answer, retrieved = gtAnswer(query, topk)

    recall, precision, total_relevant = evaluate_recall_precision(
        retrieved, ground_truth, chunks
    )

    hallucinated, missing = hallucination_check(answer, retrieved)

    return {
        "query": query,
        "answer": answer,
        "recall@k": recall,
        "precision@k": precision,
        "total_relevant_passages": total_relevant,
        "hallucination": hallucinated,
        "missing_terms": missing,
        "retrieved_chunks": retrieved
    }

In [ ]:
ground_truth = "For CRAG go "
result = evaluate_rag("For CRAG where to go", ground_truth, topk=5)
print("Recall@k:", result["recall@k"])
print("Precision@k:", result["precision@k"])
print("Hallucination:", result["hallucination"])
print("Answer:", result["answer"])

Recall@k: 1.0
Precision@k: 0.2
Hallucination: False
Answer: slides
